# StruQ Defense: Step-by-Step Structured Query Fine-Tuning & Evaluation

This notebook demonstrates how to build, fine-tune, load, and evaluate **StruQ** (*StruQ: Defending Against Prompt Injection with Structured Queries*, Chen et al., USENIX Security 2025) within the `ipi` benchmark framework.

## 📌 Overview & Workflow
1. **Environment & Dependency Setup**: Install `ipi` framework, HuggingFace `transformers`, `peft`, `trl`, `datasets`, and `bitsandbytes`.
2. **Dataset Setup & Anti-Instruction Data Construction**: Download clean instruction dataset (`alpaca_data_cleaned.json`) directly in notebook and generate StruQ anti-instruction tuning data with structured delimiters (`[MARK] [INST][COLN]` vs `[MARK] [INPT][COLN]`).
3. **Tokenizer Resizing & Embedding Warm-Start**: Enlarge vocabulary with 5 special delimiter tokens (`[INST]`, `[INPT]`, `[RESP]`, `[MARK]`, `[COLN]`) and initialize their embeddings with corresponding textual token embeddings (`instruction`, `input`, `response`, `###`, `:`).
4. **Supervised Fine-Tuning (SFT / QLoRA)**: Fine-tune with `transformers.Trainer` + `DataCollatorForStruQDataset` (the paper's own recipe) and 4-bit NormalFloat quantization (BitsAndBytes) on Kaggle T4 GPU (~16GB VRAM) or full precision.
5. **Save, Reload & Export Model Adapters**: Save fine-tuned adapters to local disk and reload them into `LocalLLM`.
6. **Benchmark Evaluation**: Wrap reloaded model into `StruQDefense` and run comparative evaluations against `ipi` prompt injection attacks.

In [ ]:
# Cell 1 — Installation & Environment Setup
!pip install -q git+https://github.com/alirezaAalaie/IPI-Aaptive.git
!pip install -q trl peft transformers datasets bitsandbytes accelerate

# Kaggle ships a pinned torchao (e.g. 0.10.0) that is too old for the peft
# version installed above. peft eagerly version-checks torchao on import
# (even though StruQ's LoRA adapters don't use torchao at all) and raises
# "Found an incompatible version of torchao..." the moment PeftModel.from_pretrained()
# is called — which silently aborts adapter loading in LocalLLM. Since we don't
# need torchao here, remove it so peft's is_torchao_available() check is skipped.
!pip uninstall -y -q torchao

import os
import json
import urllib.request
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 2 — Dataset Setup & StruQ Anti-Instruction Data Construction
os.makedirs("data", exist_ok=True)

# alpaca_data_cleaned.json -> the clean SFT corpus.
# alpaca_data.json (Stanford) -> reference responses used to build the fake
#   "### response:" block in the Completion attack (upstream struq.py:43).
DATA_URLS = {
    "data/alpaca_data_cleaned.json": "https://raw.githubusercontent.com/gururise/AlpacaDataCleaned/refs/heads/main/alpaca_data_cleaned.json",
    "data/alpaca_data.json":         "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/refs/heads/main/alpaca_data.json",
}
for path, url in DATA_URLS.items():
    if not os.path.exists(path):
        print(f"Downloading {path} ...")
        urllib.request.urlretrieve(url, path)
    print(f"✅ {path} ({os.path.getsize(path)} bytes)")

with open("data/alpaca_data_cleaned.json") as f:
    clean_samples = json.load(f)
with open("data/alpaca_data.json") as f:
    ref_inst_resp = {s["instruction"]: s["output"] for s in json.load(f)}

print(f"Loaded {len(clean_samples)} clean samples, {len(ref_inst_resp)} reference responses.")

from ipi.defenses.struq import generate_struq_training_data, format_struq_prompt, STRUQ_DELIMITERS

# NaiveCompletion = the paper's best-performing training attack mix.
# downsample=True reproduces the paper's balance: the corpus stays at its
# original size and ends up ~50% clean / ~25% Naive / ~25% Completion.
struq_training_data = generate_struq_training_data(
    clean_samples=clean_samples,          # slice this for a quick smoke test
    attack_type="NaiveCompletion",
    delimiter_scheme="SpclSpclSpcl",
    downsample=True,
    ref_inst_resp=ref_inst_resp,
    seed=42,
)

print(f"✅ Generated {len(struq_training_data)} StruQ training samples.")
print("\n--- SAMPLE STRUCTURED QUERY ---")
print(struq_training_data[1]["prompt"])
print("--- TARGET OUTPUT (clean task only) ---")
print(struq_training_data[1]["output"])


In [ ]:
# Cell 3 — Tokenizer Special Token Resizing & Embedding Warm-Start
import transformers
from ipi.defenses.struq import SPECIAL_DELM_TOKENS, TEXTUAL_DELM_TOKENS, STRUQ_DELIMITERS

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./struq_lora_weights"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Delimiter tokens:", SPECIAL_DELM_TOKENS)
print("Warm-started from:", TEXTUAL_DELM_TOKENS)
print("Instruction delimiter:", repr(STRUQ_DELIMITERS["SpclSpclSpcl"][0]))

# Before the resize the delimiter shreds into ordinary subwords. train_struq()
# performs the resize BEFORE tokenizing the corpus, so training and inference
# agree on the token ids; if you ever tokenize by hand, keep that order.
probe = STRUQ_DELIMITERS["SpclSpclSpcl"][0]
print("\nUnregistered tokenization:", tokenizer.tokenize(probe))
tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_DELM_TOKENS})
print("Registered tokenization:  ", tokenizer.tokenize(probe))


In [ ]:
# Cell 4 — Supervised Fine-Tuning (QLoRA)
from ipi.defenses.struq import train_struq

# Kaggle timings: 1.5B-3B ≈ 15-30 min; 7B-8B ≈ 1-2 h with 4-bit QLoRA.
#
# Deviation from the paper, on purpose: upstream does full-parameter FSDP SFT on
# 4 GPUs (3 epochs, lr 2e-5, effective batch 128). QLoRA on one GPU will not
# reproduce the paper's absolute numbers. `modules_to_save` keeps embed_tokens /
# lm_head trainable, which is required — the five delimiter tokens are new rows
# in the embedding matrix and LoRA on the projections cannot reach them.

"""
trainer = train_struq(
    model_name_or_path=MODEL_ID,
    output_dir=OUTPUT_DIR,
    train_samples=clean_samples,
    ref_inst_resp=ref_inst_resp,
    attack_type="NaiveCompletion",
    delimiter_scheme="SpclSpclSpcl",
    downsample=True,
    use_4bit=True,
    lora_r=16,
    lora_alpha=32,
    learning_rate=2e-4,
    num_train_epochs=3.0,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_length=512,
)
print(f"✅ Training complete! Adapter + resized tokenizer saved to {OUTPUT_DIR}")
"""
print("Fine-tuning configuration ready.")


In [ ]:
# Cell 5 — Save, Export & Reload Model Adapters
"""
# Step 1: Save / Reload Adapter into LocalLLM
from ipi.llm_unified import LocalLLM

reloaded_victim = LocalLLM(
    model=MODEL_ID,
    adapter_path=OUTPUT_DIR,
)
print("✅ Successfully reloaded StruQ fine-tuned adapter!")

# Step 2 (Optional): Push Adapter to Hugging Face Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id="your-username/StruQ-Qwen2.5-1.5B-LoRA",
    repo_type="model"
)
"""
print("Adapter saving, reloading, and publishing script ready.")


In [ ]:
# Cell 6 — Evaluate the StruQ Defense on the ipi Attack Benchmark
from ipi.target import LocalLLM, TargetLLM
from ipi.defenses.struq import StruQDefense
from ipi.attacks import NaiveAttacker, IgnoreAttacker, FakeCompletionAttacker, EscapeAttacker
from ipi.metrics import AttackEvaluator
from ipi.datasets import DualVerifiableDataset

"""
# 1. Load the fine-tuned model and wrap it in the defense.
#    StruQDefense splits every prompt into an instruction channel and a data
#    channel, strips the delimiter tokens from the data channel
#    (apply_defensive_filter), and hands the model a raw Alpaca-format string
#    rather than a chat-templated one — which is what it was fine-tuned on.
base_llm = LocalLLM(model=MODEL_ID, adapter_path=OUTPUT_DIR,
                    temperature=0.0, max_tokens=500)
struq_target = StruQDefense(
    target=TargetLLM(base_llm),
    delimiter_scheme="SpclSpclSpcl",     # must match training
    apply_defensive_filter=True,         # set False to isolate the filter's contribution
)

dataset = DualVerifiableDataset().subset(10, seed=42)

for attacker in [NaiveAttacker(), EscapeAttacker(), IgnoreAttacker(), FakeCompletionAttacker()]:
    result = AttackEvaluator(target=struq_target, attacker=attacker).run(dataset)
    print(f"{type(attacker).__name__:<24} ASR: {result.asr * 100:5.1f}%   utility: {result.utility_rate * 100:5.1f}%")
"""
print("StruQ evaluation pipeline configured.")
